# NB09_fPINN_TiempoC — repeated timing of the fixed-cost C kernels

Checks the reliability of the C timings of NB07 (0.03–0.13 µs per update, close to the timer resolution).
Each fixed-cost kernel of `NB07_fPINN_memkernels.c` (Oustaloup 5–11 modes, sum of exponentials 19/30/43 modes, window 0.5/2 s + tail) is timed:

* as a loop of `R` updates with **precomputed inputs** (no `sin()` per update, unlike NB07), after 2·10⁴ warm-up updates, repeated `NREP` times → min, quartiles, median, max;
* against the same loop **without kernel** (loop overhead);
* in 2·10⁴ consecutive **batches of 100 updates** → p50, p99, p99.9 and max time per update (real-time jitter).

Also reports `clock_getres` and the cost of one `clock_gettime` call.

**Requirements:** `gcc` (Linux/macOS, or MSYS2/MinGW on Windows). **Runtime:** QUICK ≈ 10 s; FULL ≈ 2 min.
**One switch:** `QUICK = True` checks the pipeline; `QUICK = False` runs the full set used in the paper (R = 10⁶, NREP = 31).
Output: `NB09_fPINN_TimingC_Repetido.csv` (or `_QUICK.csv`). The published values come from a cloud Intel Xeon @ 2.10 GHz, gcc 13.3 `-O2`.

In [ ]:
QUICK = True   # False = full run used in the paper (about 2 min)
R, NREP = (100_000, 5) if QUICK else (1_000_000, 31)

In [ ]:
LIB = r'''
/* NB07_fPINN_memkernels.c - memory term D^alpha q of the fPINN (3 joints, double): full L1 (naive), window + tail,
   Oustaloup (parallel Tustin modes) and sum of exponentials (SOE). Update functions use no dynamic memory.
   Kernels of NB07_fPINN_memkernels.c without main(), included by NB09_fPINN_timing_bench.c */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <string.h>
#define NJ 3
static double now_s(void){ struct timespec t; clock_gettime(CLOCK_MONOTONIC, &t); return t.tv_sec + 1e-9*t.tv_nsec; }
static double l1w(double a, long k){ return pow(k + 1.0, 1 - a) - pow((double)k, 1 - a); }

/* ---- full-memory L1 (naive): D = c * sum_{k=0}^{n-1} b_k dq_{n-k} ---- */
typedef struct { double c, *b, *dq, ql[NJ]; long n; int started; } L1full;
void l1_update(L1full *s, const double q[NJ], double D[NJ]) {
    int j; long k;
    if (!s->started) { for (j = 0; j < NJ; j++) { s->ql[j] = q[j]; D[j] = 0; } s->started = 1; return; }
    s->n++; for (j = 0; j < NJ; j++) { s->dq[NJ*s->n + j] = q[j] - s->ql[j]; s->ql[j] = q[j]; }
    for (j = 0; j < NJ; j++) { double acc = 0; for (k = 0; k < s->n; k++) acc += s->b[k]*s->dq[NJ*(s->n - k) + j]; D[j] = s->c*acc; }
}
/* ---- window of Nw increments + O(1) tail correction (mean-value form) ---- */
#define NW_MAX 5000
typedef struct { double c, a, bw[NW_MAX + 1], buf[NJ][NW_MAX], qb[NJ][NW_MAX + 1], q0[NJ], ql[NJ], ssum[NJ]; long n; int Nw, started, tail; } Win;
void win_update(Win *s, const double q[NJ], double D[NJ]) {
    int j, k, Nw = s->Nw;
    if (!s->started) { for (j = 0; j < NJ; j++) { s->q0[j] = s->ql[j] = q[j]; s->qb[j][0] = 0; D[j] = 0; s->ssum[j] = 0; } s->started = 1; return; }
    s->n++; long n = s->n; int i = (int)((n - 1) % Nw); int m = n < Nw ? (int)n : Nw;
    for (j = 0; j < NJ; j++) {
        s->buf[j][i] = q[j] - s->ql[j]; s->ql[j] = q[j];
        double acc = 0; for (k = 0; k < m; k++) { int idx = i - k; if (idx < 0) idx += Nw; acc += s->bw[k]*s->buf[j][idx]; }
        if (s->tail && n > Nw) {
            if (n - Nw - 1 >= 1) s->ssum[j] += s->qb[j][(n - Nw - 1) % (Nw + 1)];   /* read q~_{n-Nw-1} before its slot is overwritten */
            acc += s->bw[Nw]*s->qb[j][(n - Nw) % (Nw + 1)];
            if (n - Nw - 1 >= 1) acc -= (s->bw[Nw] - l1w(s->a, n - 1))*s->ssum[j]/(double)(n - Nw - 1);
        }
        s->qb[j][n % (Nw + 1)] = q[j] - s->q0[j];
        D[j] = s->c*acc;
    }
}
/* ---- Oustaloup, parallel first-order modes (Tustin) ---- */
#define NM_MAX 15
typedef struct { int nm; double K, r[NM_MAX], A[NM_MAX], Bd[NM_MAX], X[NM_MAX][NJ], up[NJ], q0[NJ]; int started; } Oust;
void oust_update(Oust *s, const double q[NJ], double D[NJ]) {
    int j, k; if (!s->started) { for (j = 0; j < NJ; j++) { s->q0[j] = q[j]; s->up[j] = 0; } s->started = 1; }
    for (j = 0; j < NJ; j++) {
        double u = q[j] - s->q0[j], acc = u;
        for (k = 0; k < s->nm; k++) { s->X[k][j] = s->A[k]*s->X[k][j] + s->Bd[k]*(u + s->up[j]); acc += s->r[k]*s->X[k][j]; }
        s->up[j] = u; D[j] = s->K*acc;
    }
}
void oust_init(Oust *s, double a, int nm, double wb, double wh, double h) {
    int N = (nm - 1)/2, i, k; double p[NM_MAX], z[NM_MAX];
    s->nm = nm; s->K = pow(wh, a); s->started = 0;
    for (i = 0; i < nm; i++) { int kk = i - N; p[i] = wb*pow(wh/wb, (kk + N + (1 + a)/2)/(2.0*N + 1)); z[i] = wb*pow(wh/wb, (kk + N + (1 - a)/2)/(2.0*N + 1)); }
    for (i = 0; i < nm; i++) { double num = 1, den = 1; for (k = 0; k < nm; k++) { num *= z[k] - p[i]; if (k != i) den *= p[k] - p[i]; }
        s->r[i] = num/den; s->A[i] = (2 - p[i]*h)/(2 + p[i]*h); s->Bd[i] = h/(2 + p[i]*h); for (k = 0; k < NJ; k++) s->X[i][k] = 0; }
}
/* ---- sum of exponentials: S_i <- E_i S_i + G_i dq_{n-1};  D = c dq_n + sum_i w_i S_i ---- */
#define NS_MAX 80
typedef struct { const char *label; int nm; const double *E, *G, *W; } SoeDef;
typedef struct { int nm; double c; const double *E, *G, *W; double S[NS_MAX][NJ], dp[NJ], ql[NJ]; int started; } Soe;
void soe_init(Soe *s, const SoeDef *d, double c) { int i, j; s->nm = d->nm; s->c = c; s->E = d->E; s->G = d->G; s->W = d->W; s->started = 0;
    for (i = 0; i < NS_MAX; i++) for (j = 0; j < NJ; j++) s->S[i][j] = 0; for (j = 0; j < NJ; j++) s->dp[j] = 0; }
void soe_update(Soe *s, const double q[NJ], double D[NJ]) {
    int j, i; if (!s->started) { for (j = 0; j < NJ; j++) { s->ql[j] = q[j]; D[j] = 0; } s->started = 1; return; }
    for (j = 0; j < NJ; j++) {
        double d = q[j] - s->ql[j], acc = s->c*d; s->ql[j] = q[j];
        for (i = 0; i < s->nm; i++) { s->S[i][j] = s->E[i]*s->S[i][j] + s->G[i]*s->dp[j]; acc += s->W[i]*s->S[i][j]; }
        s->dp[j] = d; D[j] = acc;
    }
}
static const double SOE0_E[] = {9.99999999956366792e-01, 9.99999999806395201e-01, 9.99999999140956941e-01, 9.99999996188343898e-01, 9.99999983087317967e-01, 9.99999924956816333e-01, 9.99999667026269190e-01, 9.99998522564596648e-01, 9.99993444494636674e-01, 9.99970912924223243e-01, 9.99870944317754917e-01, 9.99427494600489075e-01, 9.97462237001803920e-01, 9.88788711181735835e-01, 9.51204200893127849e-01, 8.00937441626903279e-01, 3.73471755644566694e-01, 1.26493100541422068e-02, 3.79043689969081544e-09};
static const double SOE0_G[] = {9.99999469577185218e-01, 9.99999916022709878e-01, 1.00000000083980023e+00, 9.99999996455063545e-01, 9.99999972893711564e-01, 9.99999888189260888e-01, 9.99999500605238567e-01, 9.99997783821584774e-01, 9.99990166768686017e-01, 9.99956369740791384e-01, 9.99806423416881640e-01, 9.99141378476155917e-01, 9.96196039618091334e-01, 9.83235497813260473e-01, 9.27803326978348331e-01, 7.18272320651828888e-01, 2.37574920429686892e-01, 2.85786473241332476e-03, 1.95476200358572207e-10};
static const double SOE0_W[] = {7.00535399571796779e-05, 1.47563679232881118e-04, 3.10834248234345581e-04, 6.54754139891910026e-04, 1.37920124999348559e-03, 2.90520666016348813e-03, 6.11964768615033770e-03, 1.28906794535909933e-02, 2.71534613260987663e-02, 5.71971760404411181e-02, 1.20482501575471940e-01, 2.53789333508704695e-01, 5.34592367858877648e-01, 1.12608751448239808e+00, 2.37203739992018114e+00, 4.99655786451581552e+00, 1.05249565180949247e+01, 2.21702045110856112e+01, 4.67002374041473232e+01};
static const double SOE1_E[] = {9.99999999999563682e-01, 9.99999999998715139e-01, 9.99999999996216471e-01, 9.99999999988858801e-01, 9.99999999967192799e-01, 9.99999999903393166e-01, 9.99999999715524002e-01, 9.99999999162309305e-01, 9.99999997533269513e-01, 9.99999992736269272e-01, 9.99999978610640716e-01, 9.99999937015192297e-01, 9.99999814529936137e-01, 9.99999453850194175e-01, 9.99998391764688965e-01, 9.99995264269771811e-01, 9.99986054856251805e-01, 9.99958936577169277e-01, 9.99879086206421408e-01, 9.99643989483838813e-01, 9.98952025968049107e-01, 9.96917195800435474e-01, 9.90949314314731988e-01, 9.73582503567706259e-01, 9.24190512017453836e-01, 7.92827317597911807e-01, 5.04792974619926338e-01, 1.33586881475561803e-01, 2.66473835566004595e-03, 2.62650156588574597e-08};
static const double SOE1_G[] = {9.99966391851693559e-01, 1.00000176427240750e+00, 1.00000832424466402e+00, 1.00000100571033168e+00, 9.99999186308905830e-01, 1.00000062044762639e+00, 1.00000002544356748e+00, 1.00000007214648789e+00, 1.00000000081651885e+00, 9.99999983121498515e-01, 9.99999966402308371e-01, 9.99999905904966457e-01, 9.99999721914338835e-01, 9.99999180759656436e-01, 9.99997587661823784e-01, 9.99992896426711630e-01, 9.99979082365855865e-01, 9.99938405567823652e-01, 9.99818635401319877e-01, 9.99466037037311961e-01, 9.98428496604075177e-01, 9.95379754790179816e-01, 9.86458133679236626e-01, 9.60665316122679136e-01, 8.88699062146129815e-01, 7.07526494347540558e-01, 3.65673653786524777e-01, 5.74968910612892875e-02, 4.48345936803389424e-04, 1.50472488440240450e-09};
static const double SOE1_W[] = {5.07770625192980291e-06, 8.71337877247049202e-06, 1.49522177663752314e-05, 2.56581082919822310e-05, 4.40294898997236879e-05, 7.55549068064246973e-05, 1.29652738551562150e-04, 2.22484989055539813e-04, 3.81785767952429070e-04, 6.55146997690883798e-04, 1.12423674377736330e-03, 1.92919796704231396e-03, 3.31051694995768706e-03, 5.68086980350667337e-03, 9.74841156599619112e-03, 1.67283411426516933e-02, 2.87059481937595763e-02, 4.92596040860144371e-02, 8.45298186401099716e-02, 1.45053748845670155e-01, 2.48913228404808812e-01, 4.27136808031238546e-01, 7.32969693673341416e-01, 1.25778102411698622e+00, 2.15836086851064568e+00, 3.70376206143553555e+00, 6.35568111332331220e+00, 1.09063924043214922e+01, 1.87154442074964855e+01, 3.21158306889018021e+01};
static const double SOE2_E[] = {9.99999999999995670e-01, 9.99999999999989675e-01, 9.99999999999975575e-01, 9.99999999999942379e-01, 9.99999999999863887e-01, 9.99999999999678368e-01, 9.99999999999240052e-01, 9.99999999998204103e-01, 9.99999999995756061e-01, 9.99999999989971022e-01, 9.99999999976299958e-01, 9.99999999943992912e-01, 9.99999999867646205e-01, 9.99999999687226637e-01, 9.99999999260866357e-01, 9.99999998253308542e-01, 9.99999995872287384e-01, 9.99999990245551973e-01, 9.99999976948671931e-01, 9.99999945526008460e-01, 9.99999871269209195e-01, 9.99999695788481913e-01, 9.99999281099446913e-01, 9.99998301123302569e-01, 9.99995985286013966e-01, 9.99990512611672488e-01, 9.99977579921796922e-01, 9.99947018562064693e-01, 9.99874800869588798e-01, 9.99704159582927243e-01, 9.99301022519269333e-01, 9.98348990756209642e-01, 9.96102789425328816e-01, 9.90814717076455853e-01, 9.78429441581562753e-01, 9.49772773999353448e-01, 8.85343478500546066e-01, 7.49922988879656360e-01, 5.06575638284872509e-01, 2.00459674042174879e-01, 2.24170101268971640e-02, 1.26517332015883816e-04, 6.14828724147557646e-10};
static const double SOE2_G[] = {9.92333060616611862e-01, 1.00134226559856221e+00, 1.00237222805548298e+00, 1.00064596412299989e+00, 1.00025402116608308e+00, 1.00017288512408897e+00, 1.00001561025577801e+00, 1.00002581400378276e+00, 1.00001306383778155e+00, 9.99999993552142419e-01, 9.99998143433341369e-01, 9.99999577694308139e-01, 9.99999925168862602e-01, 1.00000017349220394e+00, 1.00000007329356566e+00, 1.00000000655053101e+00, 1.00000000859099281e+00, 9.99999984273464371e-01, 9.99999966732023404e-01, 9.99999918979292635e-01, 9.99999807192896495e-01, 9.99999543574783489e-01, 9.99998921584228095e-01, 9.99997451693168005e-01, 9.99993977927100763e-01, 9.99985768959121724e-01, 9.99966370093354984e-01, 9.99920529013850201e-01, 9.99812207836119793e-01, 9.99556275842658493e-01, 9.98951737363799275e-01, 9.97524622085092849e-01, 9.94160515045470539e-01, 9.86257261939812668e-01, 9.67838454318025732e-01, 9.25715696343551553e-01, 8.33558373236038985e-01, 6.51662374822311596e-01, 3.67539350428827405e-01, 9.97270777756146115e-02, 5.77010572070417189e-03, 1.40946490804874368e-05, 2.89881223854289712e-11};
static const double SOE2_W[] = {4.04335868209225158e-07, 6.21568355445057463e-07, 9.55510630807446882e-07, 1.46886590603911156e-06, 2.25802516514218627e-06, 3.47116617347617600e-06, 5.33607631566254852e-06, 8.20292346248004537e-06, 1.26100058077881641e-05, 1.93848263000098768e-05, 2.97994700723667608e-05, 4.58094595664976513e-05, 7.04209363682792804e-05, 1.08255114247452074e-04, 1.66415988839474412e-04, 2.55824230882208787e-04, 3.93267723629627940e-04, 6.04553766918351602e-04, 9.29354826584689766e-04, 1.42865769921322232e-03, 2.19621479669070314e-03, 3.37614771954084219e-03, 5.19000848247453024e-03, 7.97837958696336925e-03, 1.22648240457834644e-02, 1.88541930393764968e-02, 2.89837501002131528e-02, 4.45554879021959199e-02, 6.84932589930150743e-02, 1.05291777699353370e-01, 1.61860577436104075e-01, 2.48821390429520173e-01, 3.82502554457524124e-01, 5.88004929616264937e-01, 9.03915001936080764e-01, 1.38955013737439881e+00, 2.13609640302634229e+00, 3.28373026657666545e+00, 5.04793905759818173e+00, 7.75998229470602396e+00, 1.19290911651382885e+01, 1.83380851427021625e+01, 2.81903593530878496e+01};
static const SoeDef SOES[] = {{"1e-2", 19, SOE0_E, SOE0_G, SOE0_W}, {"1e-3", 30, SOE1_E, SOE1_G, SOE1_W}, {"1e-4", 43, SOE2_E, SOE2_G, SOE2_W}};
#define N_SOE 3
'''

BENCH = r'''/* NB09_fPINN_timing_bench.c - repeated timing of the fixed-cost kernels: loop timing (R updates per repetition, NREP repetitions),
   loop overhead (same input handling, no kernel), and per-batch latency distribution (batches of B updates).
   Build: gcc -O2 -std=gnu99 NB09_fPINN_timing_bench.c -o bench -lm ; Run: ./bench [R NREP] (default 1e6 31) */
#include "NB07_fPINN_memkernels_lib.h"
#include <string.h>
static long R = 1000000L; static int NREP = 31;
#define B 100
#define NB 20000
static int cmp(const void *x, const void *y){ double a=*(const double*)x, b=*(const double*)y; return (a>b)-(a<b); }
static double qs_(double *v, int n, double p){ qsort(v, n, sizeof(double), cmp); double r=p*(n-1); int i=(int)r; return i+1<n ? v[i]+(r-i)*(v[i+1]-v[i]) : v[n-1]; }
static double *U; static volatile double vsink;
typedef void (*upd)(void*, const double*, double*);
static void none_upd(void *s, const double *q, double *D){ (void)s; D[0]=q[0]; D[1]=q[1]; D[2]=q[2]; }
static void run(const char *name, void *st, upd f, void (*reinit)(void*), int size){
  double t[64], D[3], q[3], sink=0; long i; int r;
  for (r=0; r<NREP; r++){ if (reinit) reinit(st);
    for (i=0;i<20000;i++){ q[0]=q[1]=q[2]=U[i]; f(st,q,D); }            /* warm-up */
    double t0=now_s(); for (i=0;i<R;i++){ q[0]=U[i]; q[1]=U[i]; q[2]=U[i]; f(st,q,D); sink+=D[0]; } t[r]=1e9*(now_s()-t0)/R; }
  double mn=t[0]; for (r=1;r<NREP;r++) if (t[r]<mn) mn=t[r];
  double med=qs_(t,NREP,.5), p25=qs_(t,NREP,.25), p75=qs_(t,NREP,.75), mx=t[NREP-1];
  /* per-batch latency */
  static double bt[NB]; long k=0; if (reinit) reinit(st);
  for (int b=0;b<NB;b++){ double t0=now_s(); for (int j=0;j<B;j++,k++){ long ii=k%R; q[0]=U[ii]; q[1]=U[ii]; q[2]=U[ii]; f(st,q,D); sink+=D[0]; } bt[b]=1e9*(now_s()-t0)/B; }
  double b50=qs_(bt,NB,.5), b99=qs_(bt,NB,.99), b999=qs_(bt,NB,.999), bmx=bt[NB-1];
  vsink+=sink;
  printf("%s,%d,%.2f,%.2f,%.2f,%.2f,%.2f,%.2f,%.2f,%.2f,%.2f\n", name, size, mn, p25, med, p75, mx, b50, b99, b999, bmx);
}
static double cglob, aglob=0.5, hglob=0.002;
static Oust OO; static int om; static double owb;
static void oust_re(void *s){ oust_init((Oust*)s, aglob, om, owb, 1e3, hglob); }
static Soe SS; static int sk; static void soe_re(void *s){ soe_init((Soe*)s, &SOES[sk], cglob); }
static Win WW; static int wn; static void win_re(void *s){ Win *W=(Win*)s; long i; W->c=cglob; W->a=aglob; W->Nw=wn; W->tail=1; W->n=0; W->started=0; for(i=0;i<=W->Nw;i++) W->bw[i]=l1w(aglob,i);
  double q[3], D[3]; for(i=0;i<5000;i++){ q[0]=q[1]=q[2]=1e-3*sin(1e-3*i); win_update(W,q,D);} }
int main(int argc, char **argv){
  if (argc > 2) { R = atol(argv[1]); NREP = atoi(argv[2]); if (NREP > 64) NREP = 64; }
  struct timespec res; clock_getres(CLOCK_MONOTONIC,&res);
  long i; U=malloc(sizeof(double)*R); for(i=0;i<R;i++) U[i]=1e-3*sin(1e-3*i);
  cglob=pow(hglob,-aglob)/tgamma(2-aglob);
  double t0=now_s(); for(i=0;i<R;i++) vsink+=now_s(); double tcall=1e9*(now_s()-t0)/R;
  printf("# clock_getres(CLOCK_MONOTONIC) = %ld ns; cost of one clock_gettime call = %.1f ns\n", res.tv_nsec, tcall);
  printf("method,size,min_ns,p25_ns,median_ns,p75_ns,max_ns,batch%d_p50_ns,batch_p99_ns,batch_p999_ns,batch_max_ns\n", B);
  run("loop overhead (no kernel)", NULL, none_upd, NULL, 0);
  int ms[]={5,7,9,11}; double wbs[]={1e-2,1e-2,1e-4,1e-4}; char nm[64];
  for(int k=0;k<4;k++){ om=ms[k]; owb=wbs[k]; sprintf(nm,"Oustaloup %d [%g; 1e3]",om,owb); run(nm,&OO,(upd)oust_update,oust_re,om); }
  for(sk=0;sk<N_SOE;sk++){ sprintf(nm,"SOE %s [1 h]",SOES[sk].label); run(nm,&SS,(upd)soe_update,soe_re,SOES[sk].nm); }
  int ws[]={250,1000}; for(int w=0;w<2;w++){ wn=ws[w]; sprintf(nm,"window %g s + tail",wn*hglob); run(nm,&WW,(upd)win_update,win_re,wn); }
  return 0; }
'''


In [ ]:
import subprocess, platform, os, io
import pandas as pd
open("NB07_fPINN_memkernels_lib.h", "w").write(LIB); open("NB09_fPINN_timing_bench.c", "w").write(BENCH)
exe = "nb09_bench.exe" if os.name == "nt" else "./nb09_bench"
subprocess.run(["gcc", "-O2", "-std=gnu99", "NB09_fPINN_timing_bench.c", "-o", exe.lstrip("./"), "-lm"], check=True)
out = subprocess.run([exe, str(R), str(NREP)], capture_output=True, text=True, check=True).stdout
header = [l for l in out.splitlines() if l.startswith("#")]
df = pd.read_csv(io.StringIO(out), comment="#")
fn = "NB09_fPINN_TimingC_Repetido" + ("_QUICK" if QUICK else "") + ".csv"
with open(fn, "w") as f:
    f.write(f"# CPU: {platform.processor() or platform.machine()}; {platform.platform()}; R = {R}, NREP = {NREP}\n")
    for l in header: f.write(l + "\n")
    df.to_csv(f, index=False)
print("\n".join(header)); print("wrote", fn)
df.round(2)